# CNN Model

### Import packages and load data

In [1]:
%reload_ext autoreload
%autoreload 2
%aimport -numpy, -pandas, -matplotlib

import sys
import yaml
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import matplotlib.pyplot as plt
# use same font as latex
# plt.rc('text', usetex=True)

plt.rc('font', family='serif')

import pandas as pd
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  
from dataset.utils.fungi_vis import FungiTasticVis
# import SimpleNamespace

#  fix random seeds for reproducibility
import random
import numpy as np

import tensorflow as tf
from tqdm import tqdm

from pathlib import Path
from collections import Counter

random.seed(0)
np.random.seed(0)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
valset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='val',
        size='300',
        task='open',
        data_subset='Mini',
        transform=None,
)

trainset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='train',
        size='300',
        task='closed',
        data_subset='Mini',
        transform=None,
)

testset = FungiTasticVis(
        root=f"{os.getcwd()}/dataset/FungiTastic/",
        split='test',
        size='300',
        task='closed',
        data_subset='Mini',
        transform=None,
)

In [3]:
os.getcwd()

'/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025'

In [4]:
BASE_METADATA_PATH = f"{os.getcwd()}/baselines/closed_set/FungiTastic/metadata/FungiTastic-Mini"

train_metadata = pd.read_csv(f"{BASE_METADATA_PATH}/FungiTastic-Mini-Train.csv")
val_metadata  = pd.read_csv(f"{BASE_METADATA_PATH}/FungiTastic-Mini-ClosedSet-Val.csv")

In [5]:
train_metadata = train_metadata[['species','year','month','day','habitat','countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion']]

In [6]:
val_metadata = val_metadata[['species', 'year','month','day','habitat','countryCode','hasCoordinate','iucnRedListCategory','substrate','latitude','longitude','coorUncert','region','district','metaSubstrate','poisonous','elevation','landcover','biogeographicalRegion']]

### Data preprocessing

In this code chunk, I am defining functions that first extract the training paths and labels from the datasets loaded above, and then define a function that creates a tf.dataset so that not all images need to be loaded into memory at once (will slow computer or crash).

In [7]:
def extract_paths_and_labels(ds):
    """Extract file paths and labels."""
    paths, labels = [], []
    for i in tqdm(range(len(ds)), desc="Indexing"):
        _, y, p = ds[i]  # (PIL_image, label, path)
        
        paths.append(str(p))
        if y:
            labels.append(int(y))  
    return paths, labels


def get_top_labels(labels, top_n=20):
    label_counts = Counter(labels)
    top_labels = [label for label, _ in label_counts.most_common(top_n)]
    print(f"Top {top_n} species (labels): {top_labels}")
    return set(top_labels)


def filter_by_labels(paths, labels, metadata, allowed_labels):
    # metadata must be a numpy array or df -> convert to numpy first
    metadata_np = np.asarray(metadata, dtype=np.float32)

    filtered = []
    for i, (p, y) in enumerate(zip(paths, labels)):
        if y in allowed_labels:
            if i >= len(metadata_np):
                continue
            else:
                m = metadata_np[i]      # row i, shape (20,)
            filtered.append((p, m, y))

    f_paths, m_data, f_labels = zip(*filtered)
    return list(f_paths), np.array(m_data, dtype=np.float32), list(f_labels)

In [8]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

meta_numeric = ohe.fit_transform(train_metadata)
val_meta_numeric = ohe.transform(val_metadata)

In [9]:
# Train
train_paths, train_labels = extract_paths_and_labels(trainset)
top_labels = get_top_labels(train_labels, top_n=20)
train_paths, train_dataset, train_labels = filter_by_labels(train_paths, train_labels, meta_numeric, top_labels)

Indexing: 100%|██████████| 46842/46842 [00:48<00:00, 968.06it/s] 


Top 20 species (labels): [83, 53, 44, 33, 39, 179, 29, 109, 177, 85, 78, 8, 24, 144, 50, 1, 15, 79, 207, 2]


In [11]:
# Validation
val_paths, val_labels = extract_paths_and_labels(valset)

Indexing: 100%|██████████| 9450/9450 [00:12<00:00, 778.87it/s]


In [12]:
val_paths, val_dataset, val_labels = filter_by_labels(val_paths, val_labels, val_meta_numeric, top_labels)

In [13]:
autotune   = tf.data.AUTOTUNE
batch_size      = 64
h, w = 224, 224

def decode_and_resize_from_path(path):
    bytes_ = tf.io.read_file(path)
    img = tf.image.decode_image(bytes_, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, [h, w],
                          method=tf.image.ResizeMethod.LANCZOS5)
    img = tf.clip_by_value(img, 0.0, 255.0)
    img = tf.cast(img, tf.float32) / 255.0
    return img

In [14]:
top_labels = np.unique(train_labels)

label_map = {old: new for new, old in enumerate(top_labels)}

label_map_tf = tf.lookup.StaticHashTable(
    initializer=tf.lookup.KeyValueTensorInitializer(
        keys=tf.constant(list(label_map.keys()),   dtype=tf.int64),
        values=tf.constant(list(label_map.values()), dtype=tf.int64),
    ),
    default_value=tf.constant(-1, dtype=tf.int64)  
)

def make_supervised_ds(paths, metadata, labels, training=True, shuffle_buf=10_000):
    """
    Creates a tf.data.Dataset:
      - reads (path, label)
      - decodes/resizes image
      - (re)maps label via label_map_tf to contiguous [0..n_classes-1]
      - optional flip augmentation
      - batches & prefetches
    """
    ds = tf.data.Dataset.from_tensor_slices((paths, metadata, labels))
    if training:
        ds = ds.shuffle(shuffle_buf, reshuffle_each_iteration=True)

    def _load_and_map(p, m, y):
        x = decode_and_resize_from_path(p)                      
        y = tf.cast(y, tf.int64)
        y = label_map_tf.lookup(y)                              

        tf.debugging.assert_greater_equal(y, tf.constant(0, tf.int64),
                                          message="Label not in label_map (got -1).")
        return x, m, tf.cast(y, tf.int32)

    autotune = tf.data.AUTOTUNE
    ds = ds.map(_load_and_map, num_parallel_calls=autotune)
    # ds = ds.map(_load_and_map_v2, num_parallel_calls=autotune)

    def _normalize(x, m, y):
        x = tf.image.convert_image_dtype(x, tf.float32)  # scales to [0,1]
        return x, m, y

    ds = ds.map(_normalize, num_parallel_calls=autotune)
    ds = ds.ignore_errors()
    
    def maybe_augment_image(x, m, y, augment_prob=0.1):
        # Draw a random number in [0,1)
        rand_val = tf.random.uniform([], 0, 1)
        
        def augment_fn():
            # Apply brightness or other augmentations here
            image_aug = tf.image.random_brightness(x, max_delta=0.2)
            return tf.clip_by_value(image_aug, 0.0, 1.0)
        
        # Apply augmentation with given probability
        return tf.cond(rand_val < augment_prob, augment_fn, lambda: x), m, y
    
    if training:
        ds = ds.map(lambda x, m, y: (tf.image.random_flip_left_right(x), m, y),
                    num_parallel_calls=autotune)
        ds = ds.map(maybe_augment_image, num_parallel_calls=autotune)
    
    def _pack_inputs(x, m, y):
        return {"image": x, "metadata": m}, y

    ds = ds.map(_pack_inputs)

    batch_size = 64
    ds = ds.batch(batch_size, drop_remainder=training).prefetch(autotune)

    opts = tf.data.Options(); opts.experimental_deterministic = False
    return ds.with_options(opts)

In [ ]:
train_ds = make_supervised_ds(train_paths, train_dataset, train_labels, training=True)

In [ ]:
train_ds

In [16]:
val_ds   = make_supervised_ds(val_paths, val_dataset, val_labels, training=False)

In [17]:
#Sanity check
for images_metadata, labels in train_ds.take(1):
    print("Image batch shape:", images_metadata['image'].shape)
    print("Metadata batch shape:", images_metadata['metadata'].shape)
    print("Label batch shape:", labels.shape)
    print("dtype:", images_metadata['image'].dtype)
    print("Min pixel value:", tf.reduce_min(images_metadata['image']).numpy())
    print("Max pixel value:", tf.reduce_max(images_metadata['image']).numpy())

Image batch shape: (64, 224, 224, 3)
Metadata batch shape: (64, 48132)
Label batch shape: (64,)
dtype: <dtype: 'float32'>
Min pixel value: 0.0
Max pixel value: 1.0


### Baseline Model: Majority Class Predictor

In [21]:
majority_label = label_map[Counter(train_labels).most_common(1)[0][0]]
print("Majority class:", majority_label)

Majority class: 13


In [22]:
train_ds

<_OptionsDataset element_spec=({'image': TensorSpec(shape=(64, 224, 224, 3), dtype=tf.float32, name=None), 'metadata': TensorSpec(shape=(64, 48132), dtype=tf.float32, name=None)}, TensorSpec(shape=(64,), dtype=tf.int32, name=None))>

In [25]:
def majority_accuracy(dataset, majority_label):
    total = 0
    correct = 0
    for i, (i_m, y) in enumerate(dataset):
        y_np = y.numpy()
        pred = np.full_like(y_np, fill_value=majority_label)
        correct += (pred == y_np).sum()
        total += y_np.size
    return correct, total

maj_train_acc = majority_accuracy(train_ds, majority_label)
# maj_val_acc   = majority_accuracy(val_ds, majority_label)
# print(f"Majority baseline — train = {maj_train_acc:.3f}, val={maj_val_acc:.3f}")

In [26]:
maj_train_acc

(np.int64(1639), 17856)

### CNN baseline + metadata

In [18]:
n_classes = len(np.unique(train_labels))

# define early stopping class
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='accuracy', 
    verbose=1,
    patience=5,
    mode='max',
    restore_best_weights=True
)

In [19]:
from keras import layers, models, Input

tf.random.set_seed(1234)
np.random.seed(1234)

image_input = Input(shape=(224, 224, 3))


#-----------------------------------------------
x = layers.Conv2D(
    filters=48, kernel_size=(3, 3), strides=(1,1), padding='same', activation='relu', name='conv_1'
)(image_input)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Conv2D(64, (3, 3), activation='relu', name='conv_2')(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(128, (3, 3), activation='relu', name='conv_3')(x)
x = layers.MaxPooling2D(pool_size=(2,2))(x)
x = layers.Dropout(rate=0.4)(x)
x = layers.GlobalAveragePooling2D()(x)
#-------------------------------------------------


# Metadata branch --------------------------------
metadata_input = Input(shape=(48132,))
m = layers.Dense(512, activation='relu')(metadata_input)
m = layers.Dropout(0.3)(m)

m = layers.Dense(256, activation='relu')(m)
m = layers.Dropout(0.3)(m)

m = layers.Dense(64, activation='relu')(m)   # bottleneck
#--------------------------------------------------

# Combine branches --------------------------------
combined = layers.concatenate([x, m])
z = layers.Dense(128, activation='relu')(combined)
z = layers.Dense(64, activation='relu')(z)

# Final output softmax layer
output = layers.Dense(20, activation='softmax')(z)
#--------------------------------------------------

# Build model
model = models.Model(
    inputs={"image": image_input, "metadata": metadata_input}, outputs=output
)

model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=3, restore_best_weights=True
)

/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'image' mapping to value <KerasTensor shape=(None, 224, 224, 3), dtype=float32, sparse=False, ragged=False, name=keras_tensor> which has name 'keras_tensor'. Change the tensor name to 'image' (via `Input(..., name='image')`)
  warnings.warn(
/Users/jeremycui/Documents/UCB_MIDS/DATASCI207/fungitastic-classification-datasci207-Fall-2025/.dataset_demo_venv/lib/python3.12/site-packages/keras/src/models/functional.py:107: UserWarning: When providing `inputs` as a dict, all keys in the dict must match the names of the corresponding tensors. Received key 'metadata' mapping to value <KerasTensor shape=(None, 48132), dtype=float32, sparse=False, ragged=False, name=keras_tensor_9> which ha

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv2D)     │ (None, 224, 224,  │      1,344 │ input_layer[0][0] │
│                     │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 112, 112,  │          0 │ conv_1[0][0]      │
│ (MaxPooling2D)      │ 48)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv2D)     │ (None, 110, 110,  │     27,712 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 48132)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 55, 55,    │          0 │ conv_2[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │ 24,644,096 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv2D)     │ (None, 53, 53,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 26, 26,    │          0 │ conv_3[0][0]      │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 26, 26,    │          0 │ max_pooling2d_2[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ dropout[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │     16,448 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 192)       │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 128)       │     24,704 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │      8,256 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 20)        │      1,300 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 24,929,044 (95.10 MB)

 Trainable params: 24,929,044 (95.10 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/20
    100/Unknown 211s 2s/step - accuracy: 0.0922 - loss: 2.9366

: 